In [ ]:
! pip install nlpaug

In [ ]:
from huggingface_hub import notebook_login
# acees_token: 
notebook_login()

In [4]:
from transformers import HfArgumentParser, set_seed, T5ForConditionalGeneration, AutoTokenizer, Trainer, TrainingArguments
from datasets import load_dataset
from dataclasses import dataclass, field
# import nlpaug.augmenter.char as nac
from nltk.tokenize import RegexpTokenizer
from huggingface_hub import notebook_login
import torch 

In [25]:
@dataclass
class ModelArguments():
    model_name_or_path: str
    tokenizer_name: str

@dataclass
class DataTrainingArguments():
    max_len: int
   
@dataclass
class TrainingArguments_2():
    output_dir: str
    overwrite_output_dir: bool
    per_device_train_batch_size:int
    per_device_eval_batch_size:int
    gradient_accumulation_steps:int
    learning_rate: float
    warmup_steps: int
    logging_steps: int
    eval_strategy: str
    eval_steps: int
    num_train_epochs: int
    do_train: bool
    do_eval: bool
    fp16: bool
    max_steps: int
    seed: int
    report_to: str
    auto_find_batch_size: bool
    dataloader_num_workers: int
    dataloader_pin_memory: bool 
    disable_tqdm: bool 
    run_name: str 
    weight_decay: float
    save_strategy: str
    save_total_limit: int
    load_best_model_at_end: bool
    metric_for_best_model: str
    greater_is_better: bool
    push_to_hub: bool
    # hub_model_id: str
    # hub_strategy: str
    save_steps: int

In [26]:
args_dict = {
    "model_name_or_path": "google/byt5-base",
    "tokenizer_name": "google/byt5-base",
    "output_dir": './byt5-base-english-ocr-correction',
    "overwrite_output_dir": True,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps":4,
    "logging_steps": 1000,
    "learning_rate": 5e-4,
    "warmup_steps": 250,
    "eval_strategy": "steps",
    "dataloader_num_workers": 4,
    "dataloader_pin_memory": True,
    "eval_steps": 1000,
    "num_train_epochs": 1,
    "do_train": True,
    "do_eval": True,
    "seed":123,
    "report_to":"none",
    "fp16":False,
    "run_name":"byt5-base-english-ocr-correction-2",
    "weight_decay":0.01,
    "disable_tqdm":False,
    "auto_find_batch_size":True,
    "max_len":128,
    "max_steps": 10000,
    "save_strategy": "steps",
    "save_total_limit": 1,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "push_to_hub": True,
    # "hub_model_id": "vanwdai/byt5-base-vi-ocr-correction",
    # "hub_strategy":"every_save",
    "save_steps": 1000
}

In [27]:
parser = HfArgumentParser(
        (ModelArguments, DataTrainingArguments, TrainingArguments_2))
model_args, data_args, training_args = parser.parse_dict(args_dict)
set_seed(training_args.seed)

In [8]:
# Load pretrained model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_args.tokenizer_name if model_args.tokenizer_name else model_args.model_name_or_path,
    max_length=data_args.max_len
)
model = T5ForConditionalGeneration.from_pretrained(
    model_args.model_name_or_path
)

# overwriting the default max_length of 20 
tokenizer.model_max_length=128
model.config.max_length=128

tokenizer_config.json:   0%|          | 0.00/2.59k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/721 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [9]:
from datasets import DatasetDict, load_dataset
from huggingface_hub import HfApi
from datasets import Dataset

dataset = load_dataset("vanwdai/tonkenized_ocr_text")


README.md:   0%|          | 0.00/541 [00:00<?, ?B/s]

train-00000-of-00002.parquet:   0%|          | 0.00/32.0M [00:00<?, ?B/s]

train-00001-of-00002.parquet:   0%|          | 0.00/31.9M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/2.72M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/299954 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/14895 [00:00<?, ? examples/s]

In [ ]:
dataset['train'][0]
dataset['validation'][0]

In [28]:
training_args = TrainingArguments(**vars(training_args))
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
)

In [29]:
import os
# trainer.train(
#     model_path=model_args.model_name_or_path if os.path.isdir(
#     model_args.model_name_or_path) else None
# )
trainer.train(resume_from_checkpoint="./byt5-base-english-ocr-correction/checkpoint-3000")

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


FileNotFoundError: [Errno 2] No such file or directory: './byt5-base-english-ocr-correction/checkpoint-3000/trainer_state.json'

In [ ]:
trainer.push_to_hub("vanwdai/byt5-base-vi-ocr-correction")

In [19]:
trainer.save_model()
# For convenience, we also re-save the tokenizer to the same directory,
# so that you can share your model easily on huggingface.co/models =)
tokenizer.save_pretrained(training_args.output_dir)


OSError: [Errno 28] No space left on device

In [53]:
model_v1 = T5ForConditionalGeneration.from_pretrained('/kaggle/working/byt5-base-english-ocr-correction/checkpoint-6500').to("cpu")
tokenizer_v1 = AutoTokenizer.from_pretrained("/kaggle/working/byt5-base-english-ocr-correction/")

In [54]:
model_v1.push_to_hub("vanwdai/byt5-base-finetuned-nlpaug-ocr")
tokenizer_v1.push_to_hub("vanwdai/byt5-base-finetuned-nlpaug-ocr")

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/vanwdai/byt5-base-finetuned-nlpaug-ocr/commit/e0b3fd711c808871623a2bf1d9b117a3317c96f1', commit_message='Upload tokenizer', commit_description='', oid='e0b3fd711c808871623a2bf1d9b117a3317c96f1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/vanwdai/byt5-base-finetuned-nlpaug-ocr', endpoint='https://huggingface.co', repo_type='model', repo_id='vanwdai/byt5-base-finetuned-nlpaug-ocr'), pr_revision=None, pr_num=None)

In [ ]:
trainer.push_to_hub("vanwdai/byt5-base-finetuned-nlpaug-ocr")

In [ ]:
vars(training_args)

# **TEST MODEL**

In [3]:
from transformers import T5ForConditionalGeneration, AutoTokenizer

corrected_text = "tôi là người việt nam"
# augmented_text = aug.augment(corrected_text)
# print(augmented_text

augmented_text = "TAPHON PHUM 621 ĐƯỜNG HUỲNH VĂN LUỲ, P. PHÚ MỸ, TP. THỦ ĐẦU MỘT, BD DRO94610381"

model_v2 = T5ForConditionalGeneration.from_pretrained("vanwdai/byt5-base-vi-ocr-correction")
tokenizer_v2 = AutoTokenizer.from_pretrained("vanwdai/byt5-base-vi-ocr-correction")
inputs = tokenizer_v2(augmented_text, return_tensors="pt", truncation=True, max_length=256)
print("Tokenized input:", tokenizer_v2.decode(inputs["input_ids"][0], skip_special_tokens=True))

output_sequences = model_v2.generate(

    input_ids=inputs["input_ids"],

    attention_mask=inputs["attention_mask"],

    max_length=256,
    
)

print(tokenizer_v2.decode(output_sequences[0], skip_special_tokens=True))

Tokenized input: TAPHON PHUM 621 ĐƯỜNG HUỲNH VĂN LUỲ, P. PHÚ MỸ, TP. THỦ ĐẦU MỘT, BD DRO94610381
taphon phum 621 đường huỲnh văn lưỲ phú phú mỸ tp thủ đầu mỘt bd dro94610381038103810381038103810381038103


In [1]:
OCR =['TAPHSA', 'BLAVIED', 'EGOCTAN', '000 NGƯỜI OVỚI THỊ DIỄNG CUDO VÒNG TƯU DIA', 'BỊA RƯỢU - BÁNH KEO - THUỐC LÁN', 'NƯỚC GIẢI KHÁT CÁC LOẠI', 'ĐỊC: 140 ĐƯỜNG MINH CẤU - TP. THÁI NGUYÊN-Đ/2.092.41 TẠ ?099 909 301']
OCR = ' '.join(OCR)
print(OCR)

TAPHSA BLAVIED EGOCTAN 000 NGƯỜI OVỚI THỊ DIỄNG CUDO VÒNG TƯU DIA BỊA RƯỢU - BÁNH KEO - THUỐC LÁN NƯỚC GIẢI KHÁT CÁC LOẠI ĐỊC: 140 ĐƯỜNG MINH CẤU - TP. THÁI NGUYÊN-Đ/2.092.41 TẠ ?099 909 301
